In [4]:
import os
import glob
import cv2
import numpy as np
import tensorflow as tf
from sklearn.utils import shuffle
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Dropout, Activation, GlobalAveragePooling2D
from tensorflow.keras.utils import to_categorical

# ----------------------------
# Configuration
# ----------------------------
IMG_SIZE = 100
DATASET_PATH = r"D:\Coding\PDL\dataset"

# ----------------------------
# Utility: Load Images Safely
# ----------------------------
def load_images(folder_path):
    images = []

    for ext in ["jpg", "jpeg", "png", "JPG", "PNG"]:
        for img_path in glob.glob(os.path.join(folder_path, f"*.{ext}")):
            img = cv2.imread(img_path, 0)
            if img is None:
                continue
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            images.append(img)

    return np.asarray(images)

# ----------------------------
# CNN Model
# ----------------------------
def build_pothole_cnn():
    model = Sequential()

    model.add(Conv2D(
        16, (8, 8),
        strides=(4, 4),
        padding="valid",
        input_shape=(IMG_SIZE, IMG_SIZE, 1)
    ))
    model.add(Activation("relu"))

    model.add(Conv2D(32, (5, 5), padding="same"))
    model.add(Activation("relu"))

    model.add(GlobalAveragePooling2D())

    model.add(Dense(512))
    model.add(Dropout(0.1))
    model.add(Activation("relu"))

    model.add(Dense(2))
    model.add(Activation("softmax"))

    return model

# ----------------------------
# Verify Dataset Path
# ----------------------------
print("Working directory:", os.getcwd())
print("Dataset exists:", os.path.exists(DATASET_PATH))

# ----------------------------
# Load Dataset
# ----------------------------
pothole_train = load_images(os.path.join(DATASET_PATH, "train", "pothole"))
normal_train  = load_images(os.path.join(DATASET_PATH, "train", "normal"))

pothole_test  = load_images(os.path.join(DATASET_PATH, "test", "pothole"))
normal_test   = load_images(os.path.join(DATASET_PATH, "test", "normal"))

# ----------------------------
# Debug Image Counts (IMPORTANT)
# ----------------------------
print("Pothole train images:", len(pothole_train))
print("Normal train images :", len(normal_train))
print("Pothole test images :", len(pothole_test))
print("Normal test images  :", len(normal_test))

if len(pothole_train) == 0 or len(normal_train) == 0:
    raise ValueError("Training images not found. Check folder names and image formats.")

# ----------------------------
# Prepare Training & Test Sets
# ----------------------------
X_train = np.concatenate((pothole_train, normal_train), axis=0)
y_train = np.concatenate((
    np.ones(len(pothole_train)),
    np.zeros(len(normal_train))
))

X_test = np.concatenate((pothole_test, normal_test), axis=0)
y_test = np.concatenate((
    np.ones(len(pothole_test)),
    np.zeros(len(normal_test))
))

# ----------------------------
# Shuffle
# ----------------------------
X_train, y_train = shuffle(X_train, y_train)
X_test, y_test = shuffle(X_test, y_test)

# ----------------------------
# Reshape & Normalize
# ----------------------------
X_train = X_train.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0
X_test  = X_test.reshape(-1, IMG_SIZE, IMG_SIZE, 1) / 255.0

y_train = to_categorical(y_train, 2)
y_test  = to_categorical(y_test, 2)

print("Training data shape:", X_train.shape)
print("Training labels shape:", y_train.shape)

# ----------------------------
# Build & Compile Model
# ----------------------------
model = build_pothole_cnn()
model.summary()

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# ----------------------------
# Train Model
# ----------------------------
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    epochs=50,
    validation_split=0.1,
    callbacks=[early_stop]
)

# ----------------------------
# Evaluate Model
# ----------------------------
loss, accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", accuracy)

# ----------------------------
# Save Model
# ----------------------------
model.save("pothole_cnn.h5")
print("Model saved as pothole_cnn.h5")


Working directory: D:\Coding\ML
Dataset exists: True
Pothole train images: 618
Normal train images : 664
Pothole test images : 40
Normal test images  : 40
Training data shape: (1282, 100, 100, 1)
Training labels shape: (1282, 2)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)                    │ (None, 24, 24, 16)          │           1,040 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_8 (Activation)            │ (None, 24, 24, 16)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 24, 24, 32)          │          12,832 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_9 (Activation)            │ (None, 24, 24, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_2           │ (None, 32)                  │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 512)                 │          16,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_10 (Activation)           │ (None, 512)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 2)                   │           1,026 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ activation_11 (Activation)           │ (None, 2)                   │               0 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 31,794 (124.20 KB)

 Trainable params: 31,794 (124.20 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 32ms/step - accuracy: 0.5153 - loss: 0.6943 - val_accuracy: 0.4806 - val_loss: 0.6935
Epoch 2/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.5238 - loss: 0.6922 - val_accuracy: 0.4806 - val_loss: 0.6932
Epoch 3/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5182 - loss: 0.6926 - val_accuracy: 0.4884 - val_loss: 0.6926
Epoch 4/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5297 - loss: 0.6923 - val_accuracy: 0.4806 - val_loss: 0.6941
Epoch 5/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5140 - loss: 0.6935 - val_accuracy: 0.4806 - val_loss: 0.6981
Epoch 6/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5131 - loss: 0.6940 - val_accuracy: 0.4806 - val_loss: 0.6967
Epoch 7/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.5231 - loss: 0.6920 - val_accuracy: 0.4806 - val_loss: 0.6943
Epoch 8/50
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5099 - loss: 0.6920 - val_accuracy: 0.4806 - v

Test Accuracy: 0.5249999761581421
Model saved as pothole_cnn.h5
